In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Five-layer architecture: interface examples

This notebook is the executable usage guide for the modular architecture. The raw and standard data layers are implemented; later sections define the intended public interfaces.

In [2]:
from pathlib import Path
import gc

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

from src.raw import RawDataManager
from src.standard import DATASET_IDS, StandardDataManager
from src.mapping import SpatiotemporalMappingManager
from src.case import PowerSystemCaseManager
from src.app import UnitCommitmentApplication

plt.ioff()  # Figures are shown only by an explicit display() call.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

data_root = project_root / "data"

In [3]:
raw_data = RawDataManager(project_root / "config/raw_data_sources.csv", data_root)
_catalog = raw_data._catalog
_check = raw_data.check()
_prepare = raw_data.prepare()
_source_id = "gem_integrated_power"
_file_path = raw_data.get_file(_source_id)
display(_prepare)

,provider,acquisition_method,file_format,local_path,detected_path,available,size_bytes,checksum,status,download_instructions
source_id,,,,,,,,,,
osm_china_pbf,Geofabrik/direct OSM export,direct,pbf,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/osm/china-latest.osm.pbf,<NA>,True,1560217727,,available,Automatic: prepare downloads the current china-latest.osm.pbf from Geofabrik.
gem_integrated_power,Global Energy Monitor,manual,xlsx,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/Global-Integrated-Power-March-2026-II.xlsx,<NA>,True,23337385,,available,"Manual: open the GEM page, select Download data, complete the official form, and save the workbook under data/ using its original or configured filename."
doe_storage,Sandia National Laboratories,manual,json,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/doe_global_energy_storage_database_2022.json,<NA>,True,7436775,,available,"Manual: open the Sandia GESDB project page, accept its terms of use, export projects to JSON, and save the file under data/ using its original or configured filename."
provincial_hourly_load,Figshare,figshare_api,xlsx,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/china_provincial_hourly_load_2015_2024.xlsx,<NA>,True,41867984,bf730a545bc297b20d6a7ff93571992a,available,Automatic: prepare resolves Data output.xlsx through the Figshare article API and downloads it.
worldpop_population,WorldPop,direct,tif,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/chn_ppp_2020_1km_aggregated_unadj.tif,<NA>,True,49445858,,available,Automatic: prepare downloads the GeoTIFF directly from the WorldPop file index.
era5_china_2024,Copernicus Climate Data Store,atlite_cds,nc,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/era5/era5_china_2024.nc,<NA>,True,12209805208,,available,"API: accept the ERA5 licence, configure ~/.cdsapirc, then run RawDataManager.prepare('era5_china_2024'); atlite submits serial monthly requests and validates the completed cutout."
province_boundaries,DataV,direct,geojson,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/china_provinces_datav_original.geojson,<NA>,True,582522,,available,Automatic: prepare downloads the original province GeoJSON; geometry cleaning belongs to the standard data layer.
offshore_wind_potential_eap,World Bank ESMAP,direct,zip,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/eap_technical_potential_offshore_wind.zip,<NA>,True,7940083,f8b8f1f089cff3e6c1e4f2ffc5bca93e,available,Automatic: prepare downloads the CC BY 4.0 regional Shapefile archive; spatial standardization filters ISO_Ter1=CHN and dissolves both foundation zones into one marine zone.
china_eez_marine_regions,Marine Regions / VLIZ,direct,geojson,/Users/jiang/Documents/Codex/nature-plan-paper/GridX/data/china_eez_v12.geojson,<NA>,True,2387518,,available,"Automatic: prepare queries the official Marine Regions WFS for sovereign code CHN and pol_type=200NM. Use as a reproducible modelling boundary, not as an adjudication of disputed maritime claims."


In [7]:
standard_manager = StandardDataManager(
    project_root / "config/standard_data.toml", raw_data=raw_data
)
_check = standard_manager.check()
_build = standard_manager.build()
_id = "storage"
standard_data = standard_manager.load(_id)
_schema = standard_manager.schema(_id)
display(_schema)

,dataset_id,component,role,name,dimensions,dtype,size,required,description,value
0,storage,data,dimension,row,<NA>,<NA>,474,False,Number of records in the table.,NaN
1,storage,data,column,uid,row,large_string[pyarrow],474,True,Globally unique identifier for the standardized record.,NaN
2,storage,data,column,class,row,large_string[pyarrow],474,True,Primary standardized category.,NaN
3,storage,data,column,subclass,row,large_string[pyarrow],474,True,"More specific, extensible standardized category.",NaN
4,storage,data,column,status,row,large_string[pyarrow],474,True,Lifecycle or operating status.,NaN
5,storage,data,column,power_capacity_mw,row,double[pyarrow],474,True,Storage charge/discharge power capacity in MW.,NaN
6,storage,data,column,energy_capacity_mwh,row,double[pyarrow],474,True,Storage energy capacity in MWh.,NaN
7,storage,data,column,duration_h,row,double[pyarrow],474,True,Storage duration in hours.,NaN
8,storage,data,column,voltage_kv,row,list<element: double>[pyarrow],474,True,Voltage level in kV; electrical network components use one scalar level.,NaN
9,storage,data,column,geometry,row,geometry,474,True,Canonical geometry in the dataset CRS.,NaN


In [5]:
_para = standard_data.parameter
_count = _para.count()
_rules = _para.matching_rules
resolved, candidates = _para.resolve(
    standard_data.generator.iloc[10000],
    ["minimum_output_pu", "ramp_up_pu_per_hour"],
    dataset_id="generator",
    include_candidates=True,
)
display(_rules)

,rank,criterion,direction,description,configured_value
0,-1,ineligible,excluded,No named candidate passes all applicability constraints.,<NA>
1,0,eligible,accepted,"A single named candidate passes dataset, asset selectors, capacity, voltage, time, location, scenario, and selector_json constraints.",<NA>
2,1,exact uid,preferred,Prefer applies_to_uid equal to the target asset uid.,<NA>
3,2,specificity,higher first,Prefer the candidate matching more explicit conditions.,<NA>
4,3,priority,lower first,Apply the optional per-call source priority.,<NA>
5,4,quality,configured order,Apply quality_order from standard_data.toml.,official_observation > source > standard_type_proxy > derived > generic > interpolated > proxy > low > model_default
6,5,observed time,latest first,Prefer the most recently observed eligible record.,<NA>
7,6,equal values,equivalent,Equal-rank candidates have equivalent values and units.,<NA>
8,7,ambiguous,unresolved,Equal-rank candidates conflict in value or unit.,<NA>


In [ ]:
mapping_manager = SpatiotemporalMappingManager(
    project_root / "config/mapping.toml", standard_manager
)
_check = mapping_manager.check()
_build = mapping_manager.build()
mapped_data = mapping_manager.load()
_id = "network"
_schema = mapping_manager.schema(_id)
display(mapped_data.network.bus.head(10))


In [ ]:
case_manager = PowerSystemCaseManager(
    project_root / "config/case.toml", mapped_data=mapped_data
)
_check = case_manager.check()
_build = case_manager.build()
case_data = case_manager.load()
display(case_data.generator.parameter.head(10))


In [ ]:
_plot_crs = mapped_data.config["general"]["metric_crs"]

# Managers and loaded data objects expose the same plot(dataset_id, **kwargs) API.
# The manager reuses its cached complete case instead of loading it for every plot.
def _display_and_close(_figures):
    if not isinstance(_figures, dict):
        _figures = {"figure": _figures}
    for _figure in _figures.values():
        display(_figure)
        plt.close(_figure)
    del _figures
    gc.collect()

for _id in ("spatial", "population", "generator", "storage", "network"):
    _display_and_close(
        case_manager.plot(_id, year=2024, map_crs=_plot_crs)
    )

# Plot time-series classes one at a time to avoid retaining several large maps.
for _id, _dataset in (("load", case_data.load), ("resource", case_data.resource)):
    for _class_name in _dataset["class"].values.astype(str):
        _display_and_close(
            case_manager.plot(
                _id,
                year=2024,
                class_name=_class_name,
                map_crs=_plot_crs,
            )
        )


In [ ]:
uc_application = UnitCommitmentApplication(
    case_data, project_root / "config/uc.toml"
)
# uc_result = uc_application.run()  # Rebuild only after selecting an application-scale case.
uc_result = uc_application.load()
display(uc_result.summary.to_frame())

uc_figure = uc_result.plot(
    start="2024-07-15 00:00",
    end="2024-07-21 23:00",
    title="China 900 kV+ continuous UC/ED",
)
display(uc_figure)
plt.close(uc_figure)
